# KeywordIQ — Model Training on Google Colab

**Course:** MAI417-3 Deep Learning | **Programme:** MSAIM | **University:** Christ University, Bangalore

This notebook trains the KeywordIQ multi-label keyword classifier.
Run all cells in order. GPU runtime recommended:
**Runtime → Change runtime type → T4 GPU**.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 1: Check GPU availability
# ──────────────────────────────────────────────────────────────
import tensorflow as tf

print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU Available:', gpus)
if not gpus:
    print('\n⚠️  No GPU detected!')
    print('Go to: Runtime → Change runtime type → Hardware accelerator → GPU')

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 2: Install dependencies (quiet mode)
# ──────────────────────────────────────────────────────────────
!pip install tensorflow pandas numpy scikit-learn openpyxl -q
print('✅ Dependencies installed.')

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 3: Mount Google Drive and set working directory
# ──────────────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# IMPORTANT: Change this path to match where you uploaded
# your keywordiq/ folder on Google Drive.
PROJECT_PATH = '/content/drive/MyDrive/keywordiq'

os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())
print('Files found:', os.listdir('.'))

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 4: Run preprocessing
# ──────────────────────────────────────────────────────────────
print('=' * 55)
print('  STEP 1: Running preprocessing...')
print('=' * 55)
exec(open('training/preprocess.py').read())

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 5: Run model training
# ──────────────────────────────────────────────────────────────
print('=' * 55)
print('  STEP 2: Starting model training...')
print('=' * 55)
exec(open('training/train.py').read())

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 6: Plot training curves (inline in Colab)
# ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pickle

with open('training/history.pkl', 'rb') as f:
    history = pickle.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── AUC subplot ──────────────────────────────────────────────
axes[0].plot(history['auc'], label='Train AUC', color='blue', linewidth=2)
axes[0].plot(history['val_auc'], label='Val AUC', color='orange', linewidth=2)
axes[0].set_title('AUC Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Loss subplot ─────────────────────────────────────────────
axes[1].plot(history['loss'], label='Train Loss', color='blue', linewidth=2)
axes[1].plot(history['val_loss'], label='Val Loss', color='orange', linewidth=2)
axes[1].set_title('Loss Over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Binary Cross-Entropy Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training/training_curves.png', dpi=150)
plt.show()
print('Training curves saved to training/training_curves.png')

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cell 7: Download trained files back to local machine
# ──────────────────────────────────────────────────────────────
from google.colab import files

print('Downloading trained model...')
files.download('model/keywordiq_model.h5')

print('Downloading training history...')
files.download('training/history.pkl')

print('Downloading training curves image...')
files.download('training/training_curves.png')

print('\n✅ Done! Place keywordiq_model.h5 in your local model/ folder.')

## After Downloading

1. Place `keywordiq_model.h5` in your local `model/` folder
2. Place `history.pkl` in your local `training/` folder
3. Place `training_curves.png` in your local `training/` folder
4. Run locally: `streamlit run app/streamlit_app.py`